### Imports and train - test split

In [14]:
import pandas as pd
import joblib 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier


In [16]:

data = pd.read_csv('../data/datasets/processed_dataset_v3.csv')
X = data.drop(['label'],axis=1)
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 80-20 split
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, '../models/scaler.pkl')

['../models/scaler.pkl']

#### Logistic Regression

Training and fitting/ pickel dump/ accuracy score

In [17]:
lr = LogisticRegression(max_iter=500)
lr.fit(X_train_scaled, y_train)
joblib.dump(lr,'../models/logistic.pkl')

lr_pred = lr.predict(X_test_scaled)
lr_acc = accuracy_score(y_test, lr_pred)
print(f"LogisticRegression Accuracy: {lr_acc * 100:.2f}%")
print("\nLogisticRegression Classification Report:\n", classification_report(y_test, lr_pred))

LogisticRegression Accuracy: 82.55%

LogisticRegression Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.87      0.80      7840
           1       0.90      0.79      0.84     11343

    accuracy                           0.83     19183
   macro avg       0.82      0.83      0.82     19183
weighted avg       0.84      0.83      0.83     19183



Confusion Matrix and heatmap


In [4]:
lrcm = confusion_matrix(y_test, lr_pred)
plt.figure(figsize=(6,4))
sns.heatmap(lrcm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix for LogisticRegression\nOverall Accuracy: {lr_acc * 100:.2f}%')
plt.savefig("../data/outputs/Logistic_Regression.png")
plt.close()

#### Random Forest

Training and fitting/ pickel dump/ accuracy score

In [21]:
rf = RandomForestClassifier(n_estimators= 31, max_depth=21)
rf.fit(X_train_scaled,y_train)
joblib.dump(rf,'../models/random_forest.pkl')

rf_pred = rf.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_pred)
print(f"RandomForest Accuracy: {rf_acc * 100:.2f}%")
print("\nRandomForestClassification Report:\n", classification_report(y_test, rf_pred))

RandomForest Accuracy: 91.89%

RandomForestClassification Report:
               precision    recall  f1-score   support

           0       0.87      0.94      0.90      7840
           1       0.96      0.90      0.93     11343

    accuracy                           0.92     19183
   macro avg       0.91      0.92      0.92     19183
weighted avg       0.92      0.92      0.92     19183



In [22]:
# Assuming 'rf' or 'best_rf' is your trained Random Forest
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importances)

                  Feature  Importance
0              url_length    0.134083
27          longest_token    0.117986
32      unique_char_ratio    0.093835
14                entropy    0.089220
10       suspicious_words    0.086270
30    max_consecutive_rep    0.072440
12            path_length    0.067187
22             path_depth    0.053360
3                     dot    0.050105
11                 digits    0.044451
23            digit_ratio    0.041110
16             paramaters    0.040088
2                has_dash    0.032114
18          special_chars    0.030839
19           brand_in_url    0.025409
20  brand_domain_mismatch    0.013015
9         redirect_symbol    0.005066
29         hex_char_count    0.001316
5      shortening_service    0.001037
31           has_fragment    0.000605
1                  has_at    0.000328
17           has_punycode    0.000126
6   double_slash_redirect    0.000010
4                  has_ip    0.000000
13         suspicious_tld    0.000000
8           

Confusion Matrix and heatmap

In [23]:
rfcm = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(6,4))
sns.heatmap(rfcm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix for RandomForest\nOverall Accuracy: {rf_acc * 100:.2f}%')
plt.savefig("../data/outputs/Random_Forest.png")
plt.close()

#### SVM
Training and fitting/ pickel dump/ accuracy score

In [24]:
svm = SVC(kernel='rbf', cache_size=2000)
svm.fit(X_train_scaled, y_train)
joblib.dump(svm,'../models/svm.pkl')

svm_pred = svm.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, svm_pred)
print(f"SVM Accuracy: {svm_acc * 100:.2f}%")
print("\nSVM Classification Report:\n", classification_report(y_test, svm_pred))

SVM Accuracy: 87.12%

SVM Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.90      0.85      7840
           1       0.93      0.85      0.89     11343

    accuracy                           0.87     19183
   macro avg       0.87      0.88      0.87     19183
weighted avg       0.88      0.87      0.87     19183



Confusion Matrix and heatmap

In [25]:
svmcm = confusion_matrix(y_test, svm_pred)
plt.figure(figsize=(6,4))
sns.heatmap(svmcm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix for SVM\nOverall Accuracy: {svm_acc * 100:.2f}%')
plt.savefig("../data/outputs/SVM.png")
plt.close()